# **Egyptian Legal Project**
- this nootbook is to make a rag pipeline for the project

### **PDF to JSON Pipeline**
- #### **Load PDF** -> **Extract Text** -> **Parse for every rule** -> **save JSON** 

### Import The dependencies

In [42]:
from langchain_community.document_loaders import PyPDFLoader

import re
import json
from pathlib import Path


#### Load PDF

In [29]:
def load_pdf(file_path):
    """Load a PDF file and return its pages as a list of dictionaries."""

    loader  = PyPDFLoader(file_path)
    pages = loader.load()
    return pages

pages = load_pdf("../data/egyptian-civil-code.pdf")
print(f"Number of pages: {len(pages)}")
print("\n===== Page 1 Sample =====\n")
print(pages[0].page_content[:500])

Number of pages: 170

===== Page 1 Sample =====

القانون المدني المصري 
قانون الإصدار 
مادة ١ 
يلغي القانون المدني المعمول به أمام المحاكم الوطنية والصادر في ٨٢ أكتوبر سنة 
٣٨٨١ والقانون المدني المعمول به أمام المحاكم المختلطة والصادر في ٨٢ يونيو 
سنة ٥٧٨١ ويستعاض عنهما بالقانون المدني المرافق لهذا القانون 
مادة ٢ 
على وزير العدل تنفيذ هذا القانون ويعمل به ابتداء من ١٥ أكتوبر سنة 
.٩٤٩١ 
نأمر بأن يبصم هذا القانون بخاتم الدولة وأن ينشر في الجريدة الرسمية وينفذ 
كقانون من قوانين الدولة. 
صدر بقصر القبة في ٩ رمضان سنة ٧٦٣١  ٦١ يوليو سنة ٨٤٩١ 
نصو


#### Extract Articles from pages

In [ ]:
def extract_articles(pages) -> list[dict]:
    """
    this is a function to extract articles from the pages of a pdf file 

    args:
        pages: list of pages from the pdf file

    returns:
        list of dictionaries contain:
            - article_number: the number of the article
            - article_text: the arabic text of the article
            - article_text_en: the english text of the article
            - is_replead: boolean value indicating whether the article is replead or not

    """

    # collect all the text from the pages into a single string 
    full_text = ''
    for page in pages:
        full_text += page.page_content + '\n'

    # split the article patteren into a list of articles
    article_pattern = r'Article\s+(\d+)'
    splits = re.split(article_pattern, full_text)

    articles = []
    seen = set()

    for i in range(1, len(splits) - 1 , 2):
        article_number = int(splits[i])

        if article_number in seen:
            continue
        seen.add(article_number)

        raw_text = splits[i+1].strip()
        raw_text = ( raw_text.split('\n\n')[0] 
                     if '\n\n' in raw_text else raw_text[:1000] )

        # Separating Arabic and English
        lines = raw_text.split('\n')

        arabic_lines = []
        english_lines = []

        for line in lines: # if there is an empty line, skip it
            if not line:
                continue

            if re.search(r'[\u0600-\u06FF]', line): # if the line contains Arabic characters, add it to arabic_lines
                arabic_lines.append(line)
            else:
                english_lines.append(line)

        article_text_ar = ' '.join(arabic_lines).strip()
        article_text_en = ' '.join(english_lines).strip()

        article_text_ar = re.sub(r'\s+', ' ', article_text_ar).strip()
        article_text_en = re.sub(r'\s+', ' ', article_text_en).strip()


        is_replead = any(
            word in raw_text 
            for word in ['ملغاة', 'ألغيت', 'Repealed', 'repealed']
        )

        if not article_text_ar and not article_text_en:
            continue

        articles.append({
            "article_number": article_number,
            "article_text": article_text_ar,
            "article_text_en": article_text_en,
            "is_repealed": is_replead
        })

    articles = sorted(articles, key=lambda x: x['article_number'])    
    return articles


articles = extract_articles(pages)

print(f"✅ Total articles : {len(articles)}")
print(f"✅ Repealed       : {sum(1 for a in articles if a['is_repealed'])}")

print("\n=== Sample Article ===")
print(json.dumps(articles[0], ensure_ascii=False, indent=2))


✅ Total articles : 1092
✅ Repealed       : 4

=== Sample Article ===
{
  "article_number": 1,
  "article_text": "مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة على هذا الإلغاء، أو يشتمل على نص يتعارض مع نص التشريع القديم ، أو ينظم من جديد الموضوع الذي سبق أن قرر قواعده ذلك التشريع .",
  "article_text_en": "Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.",
  "is_repealed": false
}


## Save JSON

In [43]:
output_path = Path("../data/egyptian-civil-code.json")
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(articles, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(articles)} articles to {output_path}")

✅ Saved 1092 articles to ..\data\egyptian-civil-code.json
